In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import torchvision.transforms.functional as TF
from PIL import Image
from skimage.measure import label
from tqdm import tqdm

train_zip_path = "/kaggle/input/competitions/data-science-bowl-2018/stage1_train.zip"
test_zip_path = "/kaggle/input/competitions/data-science-bowl-2018/stage2_test_final.zip"
train_dir = "/kaggle/working/stage1_train"
test_dir = "/kaggle/working/stage2_test_final"

os.makedirs(train_dir, exist_ok=True)
if len(os.listdir(train_dir)) == 0:
    with zipfile.ZipFile(train_zip_path, 'r') as zip_ref:
        zip_ref.extractall(train_dir)

os.makedirs(test_dir, exist_ok=True)
if len(os.listdir(test_dir)) == 0:
    with zipfile.ZipFile(test_zip_path, 'r') as zip_ref:
        zip_ref.extractall(test_dir)

In [ ]:
class DSB2018Dataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.folder_ids = os.listdir(image_dir)

    def __len__(self):
        return len(self.folder_ids)

    def __getitem__(self, idx):
        folder_id = self.folder_ids[idx]
        current_folder_path = os.path.join(self.image_dir, folder_id)
        image_folder = os.path.join(current_folder_path, "images")
        masks_folder = os.path.join(current_folder_path, "masks")

        image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
        image = Image.open(image_path).convert("RGB")

        mask_filenames = os.listdir(masks_folder)
        w, h = image.size
        master_mask = np.zeros((h, w), dtype=np.float32)

        for mask_name in mask_filenames:
            mask_path = os.path.join(masks_folder, mask_name)
            single_mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
            master_mask = np.maximum(master_mask, single_mask / 255.0)

        master_mask = Image.fromarray(master_mask)

        if self.transform is not None:
            image, master_mask = self.transform(image, master_mask)

        return image, master_mask

transform_pipeline = v2.Compose([
    v2.Resize((256, 256)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

train_dataset = DSB2018Dataset(image_dir=train_dir, transform=transform_pipeline)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return self.final_conv(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3, out_channels=1).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

NUM_EPOCHS = 40

for epoch in range(NUM_EPOCHS):
    model.train()
    loop = tqdm(train_loader, leave=True)
    epoch_loss = 0

    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device)
        targets = targets.to(device)

        predictions = model(data)
        loss = loss_fn(predictions, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] Avg Loss: {epoch_loss/len(train_loader):.4f}")

In [ ]:
def rle_encoding(mask):
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

model.eval()
submission_data = []
test_ids = os.listdir(test_dir)

with torch.no_grad():
    for image_id in tqdm(test_ids):
        image_path = os.path.join(test_dir, image_id, "images", f"{image_id}.png")
        original_image = Image.open(image_path).convert("RGB")
        original_w, original_h = original_image.size

        input_image = TF.resize(original_image, (256, 256))
        input_tensor = TF.to_tensor(input_image).unsqueeze(0).to(device)

        prediction = model(input_tensor)
        prob_mask = torch.sigmoid(prediction).squeeze().cpu()
        prob_mask = TF.resize(prob_mask.unsqueeze(0), (original_h, original_w)).squeeze()

        binary_mask = (prob_mask > 0.5).numpy()
        labeled_mask = label(binary_mask)
        num_nuclei = labeled_mask.max()

        if num_nuclei == 0:
            submission_data.append({'ImageId': image_id, 'EncodedPixels': ''})
        else:
            for i in range(1, num_nuclei + 1):
                single_nucleus_mask = (labeled_mask == i).astype(np.uint8)
                submission_data.append({'ImageId': image_id, 'EncodedPixels': rle_encoding(single_nucleus_mask)})

pd.DataFrame(submission_data).to_csv('/kaggle/working/submission.csv', index=False)